# Spotify Data Preprocessing



In [ ]:
import pandas as pd
import json
from pathlib import Path


## 1. Find the Spotify files

In [ ]:
# project folder
project = Path.cwd()
if project.name == 'notebooks':
    project = project.parent

raw = project / 'data' / 'raw' / 'spotify_extended_history'
processed = project / 'data' / 'processed'
processed.mkdir(parents=True, exist_ok=True)

files = sorted(raw.glob('Streaming_History_Audio*.json'))

print('Files found:', len(files))
for file in files:
    print(file.name)


## 2. Load all the JSON files

In [ ]:
all_data = []

for file in files:
    with open(file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    all_data.extend(data)

print('Total records:', len(all_data))


In [ ]:
df = pd.DataFrame(all_data)

print('Shape:', df.shape)
df.head()


## 3. Check the columns

In [ ]:
print(df.columns.tolist())


## 4. Keep only music tracks

In [ ]:
# Some Spotify records can be for podcasts/audiobooks.
# For this project I only need records with a song and artist.
df = df[
    df['master_metadata_track_name'].notna()
    & df['master_metadata_album_artist_name'].notna()
].copy()

print('Rows after keeping music:', len(df))


## 5. Remove columns I don't need

In [ ]:
remove = [
    'ip_addr',
    'platform',
    'conn_country',
    'episode_name',
    'episode_show_name',
    'spotify_episode_uri',
    'audiobook_title',
    'audiobook_uri',
    'audiobook_chapter_uri',
    'audiobook_chapter_title',
    'offline',
    'offline_timestamp',
    'incognito_mode'
]

# Some of these columns may not exist, so only remove the ones that do.
remove = [col for col in remove if col in df.columns]
df.drop(columns=remove, inplace=True)

print('Removed:', remove)


## 6. Make the time columns

In [ ]:
df['timestamp'] = pd.to_datetime(
    df['ts'],
    errors='coerce',#invalid date → NaT(Not a time) instead of error
    utc=True
)

df['minutes_played'] = df['ms_played'].fillna(0) / 60000

df['year'] = df['timestamp'].dt.year
df['month'] = df['timestamp'].dt.strftime('%Y-%m')
df['day_of_week'] = df['timestamp'].dt.day_name()
df['hour'] = df['timestamp'].dt.hour
df['is_weekend'] = df['timestamp'].dt.dayofweek >= 5

df[
    [
        'timestamp',
        'minutes_played',
        'year',
        'month',
        'day_of_week',
        'hour',
        'is_weekend'
    ]
].head()

## 7. Remove missing values and exact duplicates

In [ ]:
before = len(df)

df.dropna(
    subset=['timestamp', 'master_metadata_track_name', 'master_metadata_album_artist_name'],
    inplace=True
)

df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

print('Rows before:', before)
print('Rows after:', len(df))


## 8. Quick check

In [ ]:
print('Rows:', len(df))
print('Columns:', len(df.columns))
print('Unique tracks:', df['master_metadata_track_name'].nunique())
print('Unique artists:', df['master_metadata_album_artist_name'].nunique())
print('Listening hours:', round(df['minutes_played'].sum() / 60, 2))

df.head()


## 9. Save the clean file

In [ ]:
output = processed / 'spotify_clean_history.csv'
df.to_csv(output, index=False)

print('Saved to:', output)


Done. The cleaned CSV will be used in the next notebook for feature engineering.